# NLP: Speculative Decoding

## Task Description
Accelerate text generation using a small 'draft' model to predict tokens for a larger 'target' model.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives

*   **Practical:** Gain hands-on experience with this model.
*   **Theory:** Understand the underlying concepts and architecture.

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
Install necessary libraries.

In [ ]:
!pip install transformers torch accelerate bitsandbytes -q

## 2. Imports
Import libraries.

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

## 3. Load Data
Load dataset or input data.

In [ ]:
prompt = "Write a short story about a robot discovering a flower in a post-apocalyptic wasteland."
print(f"Prompt: {prompt}")

## 4. Data Cleaning
Preprocess the data.

In [ ]:
pass

## 5. Load Model
Load the model from Hugging Face.

In [ ]:
# We use a larger target model and a smaller draft model
target_model_id = "Qwen/Qwen2.5-3B-Instruct"
draft_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(target_model_id)

# Load Target (optimized for T4)
target_model = AutoModelForCausalLM.from_pretrained(
    target_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load Draft
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Models loaded.")

## 6. Inference
Run the model.

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 1. Standard Generation
print("Running Standard Generation...")
start_time = time.time()
output_standard = target_model.generate(**inputs, max_new_tokens=100)
standard_time = time.time() - start_time
print(f"Standard Time: {standard_time:.4f}s")

# 2. Speculative Decoding
print("\nRunning Speculative Decoding...")
start_time = time.time()
output_spec = target_model.generate(**inputs, max_new_tokens=100, assistant_model=draft_model)
spec_time = time.time() - start_time
print(f"Speculative Time: {spec_time:.4f}s")

# Check speedup
speedup = standard_time / spec_time
print(f"\nSpeedup: {speedup:.2f}x")
print("Generated Text:")
print(tokenizer.decode(output_spec[0], skip_special_tokens=True))

results = f"Speedup: {speedup:.2f}x"


## 7. Save Results
Save outputs.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage
Explore further.

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print('Try different inputs!')

## 9. Troubleshooting

*   **Out of Memory (OOM):** If you encounter OOM errors, try reducing the batch size or input length. On the Free Tier, you are limited to T4 GPUs with 16GB VRAM.
*   **GPU Not Found:** Ensure your runtime type is set to GPU. Go to *Runtime > Change runtime type* and select *T4 GPU*.
*   **Missing Packages:** If you see `ModuleNotFoundError`, make sure you have run the **Installation** step (Step 1).